# Exp 17 — Обзор русских датасетов

Задача: посмотреть на каждый из 6 кандидатов (см. `datasets.md`), понять
структуру, содержимое JSON-колонок и потенциал для entity resolution.

Feature engineering — **после** согласования с пользователем по каждому датасету.

Датасеты:
1. `kirill57678/lamoda-products` — товары Lamoda, JSON `About`
3. `egorledyaev/russian-car-market-dataset` — объявления авто
4. `fiftin/ozon-what-products-do-users-add-to-favs` — товары Ozon
5. `rzabolotin/auto-ru-car-ads-parsed` — авто, JSON `complectation_dict`
6. `snezhanausova/all-auto-ru-14-11-2020csv` — авто, JSON `equipment_dict`
7. `ruslanusov/dataset-of-electronics-with-lifecycle-and-specs` — устройства, JSON `technical_specs` (основной для нашей задачи)

Запуск:
1. `uv sync` — поставит `kagglehub`.
2. Один раз авторизоваться в Kaggle: в ячейке ниже запустить `kagglehub.login()` — введёт username + token (https://www.kaggle.com/settings → API → Create New Token) и сохранит их в `~/.config/kagglehub/`.
3. Дальше просто прогнать ноутбук.

In [ ]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

import kagglehub
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 240)

RAW_DIR = Path("data/raw_ru")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Если kagglehub ещё не авторизован на этой машине — раскомментировать и запустить один раз:
# kagglehub.login()

In [ ]:
def summarize(df: pd.DataFrame, name: str) -> None:
    """Сводка: размер, dtypes, доля пустых, примеры уникальных."""
    print(f"=== {name} ===")
    print(f"shape: {df.shape}")
    print(f"memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
    print()
    info = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_unique": df.nunique(dropna=True),
        "n_null": df.isna().sum(),
        "null_%": (df.isna().mean() * 100).round(1),
    })
    print(info.to_string())


def sniff_json_column(df: pd.DataFrame, col: str, n_samples: int = 50) -> Counter:
    """Посчитать, какие ключи чаще всего встречаются в JSON-колонке."""
    keys: Counter = Counter()
    for val in df[col].dropna().head(n_samples * 20):
        try:
            if isinstance(val, str):
                val = val.replace("'", '"')
                parsed = json.loads(val)
            elif isinstance(val, dict):
                parsed = val
            else:
                continue
            if isinstance(parsed, dict):
                keys.update(parsed.keys())
        except (json.JSONDecodeError, TypeError):
            continue
    return keys


def duplicate_signals(df: pd.DataFrame, subset_cols: list[str], name: str) -> None:
    """Какой процент строк — потенциальные дубли по заданному набору ключей."""
    sub = df[subset_cols].dropna(how="any")
    grouped = sub.groupby(subset_cols).size()
    n_groups = len(grouped)
    n_dup_groups = int((grouped > 1).sum())
    n_dup_rows = int(grouped[grouped > 1].sum())
    print(f"[{name}] по {subset_cols}: "
          f"{n_dup_groups}/{n_groups} групп с >1 строкой "
          f"(≈{n_dup_rows} строк, {100 * n_dup_rows / max(len(sub), 1):.1f}% от непустых)")

## 1. Lamoda products

В колонке `About` лежит dict со свойствами товара (`Состав`, `Сезон`, `Размер`…).
Для ER нужны только атрибуты товара, фото — выкидываем.

In [ ]:
lamoda_path = Path(kagglehub.dataset_download("kirill57678/lamoda-products"))
print("→", lamoda_path)
for p in sorted(lamoda_path.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(lamoda_path)}  ({p.stat().st_size / 1e6:.1f} MB)")

In [ ]:
lamoda_csvs = sorted(lamoda_path.rglob("*.csv"))
lamoda_csvs

In [ ]:
lamoda = pd.read_csv(lamoda_csvs[0])
summarize(lamoda, "Lamoda")

In [ ]:
lamoda.head(3)

In [ ]:
# Разбор JSON-колонки About
about_col = next((c for c in lamoda.columns if c.lower() == "about"), None)
print("About column:", about_col)
if about_col:
    print("\nПример сырого значения:")
    print(lamoda[about_col].dropna().iloc[0])
    print("\nЧастоты ключей (top 30):")
    for k, n in sniff_json_column(lamoda, about_col).most_common(30):
        print(f"  {n:5d}  {k}")

In [ ]:
# Потенциальные дубликаты
for subset in (["Name"], ["Name", "Brand"], ["Name", "Price"]):
    cols = [c for c in subset if c in lamoda.columns]
    if len(cols) == len(subset):
        duplicate_signals(lamoda, cols, "Lamoda")

## 3. Russian car market (egorledyaev)

In [ ]:
cars_ru_path = Path(kagglehub.dataset_download("egorledyaev/russian-car-market-dataset"))
print("→", cars_ru_path)
for p in sorted(cars_ru_path.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(cars_ru_path)}  ({p.stat().st_size / 1e6:.1f} MB)")

In [ ]:
cars_ru_csvs = sorted(cars_ru_path.rglob("*.csv"))
cars_ru_csvs

In [ ]:
# В датасете test.csv + возможно train.csv — читаем всё
cars_ru = {p.stem: pd.read_csv(p) for p in cars_ru_csvs}
for name, df in cars_ru.items():
    summarize(df, f"cars_ru/{name}")
    print()

In [ ]:
next(iter(cars_ru.values())).head(3)

In [ ]:
# Дубли: авто из одной модификации могут встречаться несколько раз
df = next(iter(cars_ru.values()))
for subset in (
    ["brand", "model"],
    ["brand", "model", "year"],
    ["brand", "model", "year", "engine_volume"],
):
    cols = [c for c in subset if c in df.columns]
    if len(cols) == len(subset):
        duplicate_signals(df, cols, "cars_ru")

## 4. Ozon favorites

In [ ]:
ozon_path = Path(kagglehub.dataset_download("fiftin/ozon-what-products-do-users-add-to-favs"))
print("→", ozon_path)
for p in sorted(ozon_path.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(ozon_path)}  ({p.stat().st_size / 1e6:.1f} MB)")

In [ ]:
ozon_files = sorted(p for p in ozon_path.rglob("*") if p.suffix.lower() in {".csv", ".json", ".parquet"})
ozon_files

In [ ]:
def _read_any(path: Path) -> pd.DataFrame:
    if path.suffix == ".csv":
        return pd.read_csv(path)
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix == ".json":
        try:
            return pd.read_json(path, lines=True)
        except ValueError:
            return pd.read_json(path)
    raise ValueError(path)

ozon = {p.stem: _read_any(p) for p in ozon_files}
for name, df in ozon.items():
    summarize(df, f"ozon/{name}")
    print()

In [ ]:
next(iter(ozon.values())).head(3)

## 5. auto.ru parsed (rzabolotin)

JSON-колонка `complectation_dict` — комплектации авто (ключ→bool/str/int).

In [ ]:
auto_ru_path = Path(kagglehub.dataset_download("rzabolotin/auto-ru-car-ads-parsed"))
print("→", auto_ru_path)
for p in sorted(auto_ru_path.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(auto_ru_path)}  ({p.stat().st_size / 1e6:.1f} MB)")

In [ ]:
auto_ru_files = sorted(p for p in auto_ru_path.rglob("*") if p.suffix.lower() in {".csv", ".json", ".parquet"})
auto_ru_files

In [ ]:
auto_ru = _read_any(auto_ru_files[0])
summarize(auto_ru, "auto_ru")

In [ ]:
auto_ru.head(3)

In [ ]:
if "complectation_dict" in auto_ru.columns:
    print("Пример сырого complectation_dict:\n")
    print(auto_ru["complectation_dict"].dropna().iloc[0][:800])
    print("\nТоп-30 ключей:")
    for k, n in sniff_json_column(auto_ru, "complectation_dict").most_common(30):
        print(f"  {n:5d}  {k}")

## 6. auto.ru 14-11-2020 (snezhanausova)

JSON-колонка `equipment_dict`.

In [ ]:
auto_ru_2020_path = Path(kagglehub.dataset_download("snezhanausova/all-auto-ru-14-11-2020csv"))
print("→", auto_ru_2020_path)
for p in sorted(auto_ru_2020_path.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(auto_ru_2020_path)}  ({p.stat().st_size / 1e6:.1f} MB)")

In [ ]:
auto_ru_2020_files = sorted(
    p for p in auto_ru_2020_path.rglob("*") if p.suffix.lower() in {".csv", ".json", ".parquet"}
)
auto_ru_2020 = _read_any(auto_ru_2020_files[0])
summarize(auto_ru_2020, "auto_ru_2020")

In [ ]:
auto_ru_2020.head(3)

In [ ]:
if "equipment_dict" in auto_ru_2020.columns:
    print("Пример сырого equipment_dict:\n")
    print(str(auto_ru_2020["equipment_dict"].dropna().iloc[0])[:800])
    print("\nТоп-30 ключей:")
    for k, n in sniff_json_column(auto_ru_2020, "equipment_dict").most_common(30):
        print(f"  {n:5d}  {k}")

## 7. DeviceStatus 15K (основной датасет)

Используем русский вариант `device_dataset_with_status_15000_ru.json`.
Вложенный `technical_specs` нужно распарсить в отдельные колонки.

In [ ]:
devices_path = Path(
    kagglehub.dataset_download(
        "ruslanusov/dataset-of-electronics-with-lifecycle-and-specs"
    )
)
print("→", devices_path)
for p in sorted(devices_path.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(devices_path)}  ({p.stat().st_size / 1e6:.1f} MB)")

In [ ]:
ru_file = next(
    p for p in devices_path.rglob("device_dataset_with_status_15000_ru.json")
)
with open(ru_file, encoding="utf-8") as f:
    raw = json.load(f)
print(f"records: {len(raw)}")
print("\nПример первой записи:")
print(json.dumps(raw[0], ensure_ascii=False, indent=2))

In [ ]:
devices = pd.json_normalize(raw, sep=".")
summarize(devices, "devices_ru")

In [ ]:
devices.head(3)

In [ ]:
# technical_specs раскрыт в plain-колонки через json_normalize (technical_specs.*)
spec_cols = [c for c in devices.columns if c.startswith("technical_specs.")]
print(f"technical_specs → {len(spec_cols)} колонок:")
for c in spec_cols:
    n_null = devices[c].isna().sum()
    print(f"  {c:45s}  null={n_null:5d} ({100*n_null/len(devices):.1f}%)  "
          f"unique={devices[c].nunique():5d}")

In [ ]:
# Дубликаты в devices
for subset in (
    ["manufacturer", "device_model"],
    ["manufacturer", "device_model", "release_year"],
    ["device_type", "manufacturer", "device_model"],
):
    cols = [c for c in subset if c in devices.columns]
    if len(cols) == len(subset):
        duplicate_signals(devices, cols, "devices_ru")

# Распределение типов
print("\nРаспределение device_type:")
print(devices["device_type"].value_counts())

## Итог

После просмотра каждого датасета нужно согласовать с пользователем:
- Какие колонки оставить для ER, какие выкинуть (id, фото, даты скрейпа, служебные).
- Какие новые признаки посчитать (feature engineering) — решаем по содержимому.
- Как получать пары-кандидаты (дубли внутри датасета / между датасетами / порча).

Дальше: отдельные `0X_prepare_*.py` скрипты под каждый датасет.